# Euler's Method in Python: Write the Loop Yourself

**Time:** about 20 minutes.
**You need:** nothing but this page. New to notebooks? Do `00-start-here.ipynb` first. Having done
`02-eulers-method.ipynb` helps, but you do not need it.

In Section 3.1 you ran Euler's method by hand on the fish pond: a table, a calculator, four
decimals carried down four rows. That was a lot of careful arithmetic for four steps.

This activity has you write the same thing in Python, the language scientists and engineers
reach for when they need exactly this. You do not need any programming experience. Python shows
up one piece at a time, only when Euler's method needs it, and there are only five pieces.

> **The main idea:** the code is the same move you made by hand: read the slope, take a step,
> move the clock forward. The computer just repeats it without getting tired.

This one works a little differently from the other activities. Here you **read the code**, and in a
couple of places you change it. Each section shows a small piece, asks you to predict what it will
print, and then you run it. Commit to a prediction first, on paper or out loud. Being wrong and
seeing why is the point!

## Run this first

The two cells below load the math and drawing tools. You never need to read or change them. Run
them once each (**Shift + Enter**) and give them a moment on the first run.

In [ ]:
%pip install -q ipywidgets

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

print('Ready.')

## 1. One step, with Python as a calculator

Here is the fish pond again:

$$u'=u\left(1-\frac{u}{K}\right), \qquad K=2+\sin(2\pi t), \qquad u(0)=1.$$

Read it as a sentence: *the fish population grows, but the growth slows as the population gets
close to this season's capacity $K$, and it reverses when there are more fish than the pond can
hold.* Here $u$ is thousands of fish and $t$ is years, so $K$ swings between $3$ in summer and $1$
in winter.

The first row of your table went like this. At $t=0$ you have $u=1$ and $K=2$, so the slope is
$1\left(1-\tfrac12\right)=0.5$. Then Euler's update with $h=0.25$:

$$u_{\text{next}}=u+h\cdot\text{slope}=1+0.25(0.5).$$

Here is that same row in Python:

```python
t = 0
u = 1
h = 0.25
slope = 0.5
u = u + h * slope
```

Two things to know before you run it.

First, `*` is multiplication. Python never multiplies two things just because they sit next to each
other, so $h\cdot\text{slope}$ has to be typed `h * slope`, and $2x$ has to be typed `2 * x`. Leave
the `*` out and Python stops with an error.

Second, and this is the big one, `=` does **not** mean "equals." It means **store**. Python works
out the right side first, then stores the result under the name on the left. So the last line reads
as: *take the current `u`, add `h` times `slope`, and store that as the new `u`.* In algebra,
$u=u+0.125$ would be impossible. In Python it is an instruction, and it is the most important line
in this whole activity.

**Predict:** the cell below prints `u` twice, once before the step and once after. What are the two
numbers?

In [ ]:
t = 0
u = 1
h = 0.25
slope = 0.5

print('before the step, u is', u)
u = u + h * slope
print('after the step, u is', u)

<details>
<summary>Check yourself after you have written something</summary>

It prints <code>1</code> and then <code>1.125</code>. The name <code>u</code> held 1, and then the
update line replaced it with 1.125. The old value is gone. That is exactly what you did on paper
when you moved down to the next row of the table: the new number became the one you worked with.
</details>

## 2. Let Python find the slope

On paper you worked out the slope fresh on every row. Python can do that part too, if you hand it
the recipe. A recipe with inputs and an output is called a **function**, and it is the same idea as
$f(t,u)$ in math: you give it a time and a population, it gives back the slope there.

```python
def f(t, u):
    K = 2 + math.sin(2 * math.pi * t)
    return u * (1 - u / K)
```

Line by line:

`def f(t, u):` starts a recipe named `f` with inputs `t` and `u`. The colon at the end is required.

The **indented** lines underneath are the recipe itself. The indent (four spaces) is how Python
knows which lines belong to `f`. Indenting is not decoration here, it is grammar.

`math.sin` and `math.pi` are sine and $\pi$ from Python's `math` toolkit, which the setup cell
loaded. Like calculus, Python's sine works in radians.

`return` hands the answer back to whoever asked.

**Predict:** the cell below asks for the slope at three rows of your table. Before running it,
work out the first one in your head, and say whether the third one is positive or negative.

1. `f(0, 1)`, the first row.
2. `f(0.25, 1.125)`, the second row.
3. `f(0.75, 1.4145)`, the winter row. What is $K$ at $t=0.75$?

In [ ]:
def f(t, u):
    K = 2 + math.sin(2 * math.pi * t)
    return u * (1 - u / K)

print(f(0, 1))
print(f(0.25, 1.125))
print(f(0.75, 1.4145))

<details>
<summary>Check yourself after you have written something</summary>

About <code>0.5</code>, <code>0.7031</code>, and <code>-0.5863</code>, the same slopes as your
table. Python prints more decimals than you carried, which is fine. The last one is negative
because at <i>t</i> = 0.75 the capacity is K = 2 + sin(1.5&pi;) = 1, and there are about 1.41
thousand fish. More fish than winter can support means the population shrinks.

The long tail on the last one, <code>-0.5863102500000001</code>, is not a mistake. Computers store decimals in binary and round in the last digit. It never matters at the
accuracy we care about.
</details>

### One typing mistake worth seeing once

In Python a power is written with two stars, so $u^2$ is `u ** 2`. The caret `^` is on your keyboard
and looks right, but in Python it means something else entirely.

**Predict:** what do you think each line prints?

In [ ]:
print(3 ** 2)
print(3 ^ 2)

<details>
<summary>Check yourself after you have written something</summary>

The first prints <code>9</code>. The second prints <code>1</code>, with no error and no warning.
(With a decimal, like <code>1.5 ^ 2</code>, Python does complain, but with whole numbers it just
quietly hands you the wrong answer.)
</details>

A mistake that crashes is annoying. A mistake that gives a believable wrong number is the dangerous
kind. If a result ever looks off, the first thing to check is every `^` in your formula.

## 3. Repeat it: the loop

Look back at what you did on paper. Every row was the same three moves:

1. find the slope where you are,
2. step `u` forward,
3. move the clock `t` forward by `h`.

Then you did it again from wherever you landed. Python's way of saying "do this four times" is a
**`for` loop**:

```python
for step in range(4):
    slope = f(t, u)
    u = u + h * slope
    t = t + h
```

`range(4)` means "four times." Python counts them as `step` = 0, 1, 2, 3, which is why your table
also started at step 0. The indented lines are what gets repeated. Anything after the loop that is
**not** indented runs once, after all four passes are done.

The order of the three lines matters. The slope is read using the current `t` and `u`, **before**
either one changes, because Euler's method always uses the slope at the start of the step. That
is the tangent line you drew in class.

**Predict:** get out your table from the 3.1 packet (the anchor problem, four steps to $u(1)$). The
cell below prints one line per row: the step number, `t`, `u`, the slope, and the new `u`. What will
the four lines say? If you do not have the packet handy, predict this much: does `u` go up on every
step?

In [ ]:
def f(t, u):
    K = 2 + math.sin(2 * math.pi * t)
    return u * (1 - u / K)

t = 0
u = 1
h = 0.25

print('step, t, u, slope, new u')
for step in range(4):
    slope = f(t, u)
    new_u = u + h * slope
    print(step, t, round(u, 4), round(slope, 4), round(new_u, 4))
    u = new_u
    t = t + h

print('after the loop: t =', t, 'and u =', round(u, 4))

<details>
<summary>Check yourself after you have written something</summary>

The new <code>u</code> column reads 1.125, 1.3008, 1.4145, 1.2679, the same four numbers as your
table, to the same four decimals. It goes up three times and down once: the last step starts in
winter, where the capacity has dropped below the population. (Here <code>round(u, 4)</code> only
tidies what gets printed. Python keeps every digit for the arithmetic.)
</details>

The computer did what you did, in the same order, and got the same digits. There is no extra
cleverness hiding in there.

**Your turn.** In the cell above, change `h = 0.25` to `h = 0.125` and `range(4)` to `range(8)`,
then run it again. Before you do, ask yourself why **both** have to change. You should end at
$t=1$ with $u$ about $1.299$, which matches the computer row in the packet.

## 4. Keep every step and draw it

The loop above overwrites `u` on every pass, so when it finishes only the last value is left. To
draw the path you need to keep all of them. Python's container for that is a **list**.

`ts = [0]` makes a list holding one number, 0. `ts.append(t)` adds the current `t` to the end of
the list. So inside the loop, every pass adds one new point to each list.

The last block of the cell, everything from `fig, ax` down, draws the picture. You do not need to
learn those lines. Copy them whenever you want a plot of your own.

The picture shows your Euler steps as dots joined by straight lines, along with the capacity
$K(t)$ as a dotted line, so you can see the season the fish are living through.

**Predict:** on which step does the Euler path turn downward? Where will the dots be relative to the
dotted capacity line at the moment it does?

In [ ]:
def f(t, u):
    K = 2 + math.sin(2 * math.pi * t)
    return u * (1 - u / K)

t = 0
u = 1
h = 0.25

ts = [t]
us = [u]
for step in range(4):
    u = u + h * f(t, u)
    t = t + h
    ts.append(t)
    us.append(u)

print('t values:', ts)
print('u values:', us)

fig, ax = plt.subplots(figsize=(8, 5))
tt = np.linspace(0, 1, 300)
ax.plot(tt, 2 + np.sin(2 * np.pi * tt), ':', color='0.45', linewidth=2,
        label='capacity K(t)')
ax.plot(ts, us, 'o-', color='tab:blue', linewidth=2, markersize=7,
        label='Euler steps, h = 0.25')
ax.set_title("u' = u(1 - u/K), K = 2 + sin(2πt), u(0) = 1: Euler with h = 0.25")
ax.set_xlabel('t (years)')
ax.set_ylabel('u (thousands of fish)')
ax.set_ylim(0.8, 3.1)
ax.grid(alpha=0.25)
ax.legend(loc='upper right')
plt.show()
plt.close(fig)

<details>
<summary>Check yourself after you have written something</summary>

The path turns down on the last step, the one that starts at <i>t</i> = 0.75. At that moment the
dot sits <strong>above</strong> the dotted capacity line: about 1.41 thousand fish in a pond that
can hold 1 thousand in winter. Whenever the population is above the capacity, the slope is
negative.
</details>

Notice what the picture lets you see that the table did not: the population climbs all spring and
summer while the capacity sits well above it, then the capacity drops underneath it and the fish
lose ground. Nobody solved this equation. There is no formula for its solution. You learned how the
pond behaves from the slope alone, one step at a time.

## 5. Turn the dial on h

Here is everything from the last section, wrapped up as one function called `euler`. Read it once.
Every line is something you have already seen. The only new one is the line that works out how
many steps it takes to walk from the start to the end.

Below it is a slider for $h$, from $0.5$ (two steps) down to $0.01$ (a hundred steps). The title
reports the step size, the number of steps, and the estimate of $u(1)$. The thin dashed curve is the
same method with $h=0.001$, a thousand steps, as a stand-in for the true solution.

**Predict before you drag:**

1. At $h=0.5$ you take only two steps. Will the estimate of $u(1)$ land above or below the
   $1.2679$ you got with $h=0.25$?
2. As $h$ gets small, does $u(1)$ keep changing forever, or does it settle on a number? If it
   settles, roughly where?

Then drag $h$ to **both ends** of its range on purpose. The ends are where the information is.

In [ ]:
from ipywidgets import interact, SelectionSlider


def f(t, u):
    K = 2 + math.sin(2 * math.pi * t)
    return u * (1 - u / K)


def euler(f, t0, u0, h, t_end):
    n = round((t_end - t0) / h)      # how many steps it takes to get there
    t = t0
    u = u0
    ts = [t]
    us = [u]
    for step in range(n):
        u = u + h * f(t, u)
        t = t + h
        ts.append(t)
        us.append(u)
    return ts, us


def explore_h(h=0.25):
    ts, us = euler(f, 0, 1, h, 1)
    fine_ts, fine_us = euler(f, 0, 1, 0.001, 1)

    fig, ax = plt.subplots(figsize=(8, 5))
    tt = np.linspace(0, 1, 300)
    ax.plot(tt, 2 + np.sin(2 * np.pi * tt), ':', color='0.45', linewidth=2,
            label='capacity K(t)')
    ax.plot(fine_ts, fine_us, '--', color='black', linewidth=1.2,
            label='very small steps, h = 0.001')
    marker = 'o' if len(ts) <= 26 else None
    ax.plot(ts, us, color='tab:blue', linewidth=2, marker=marker, markersize=6,
            label=f'Euler steps, h = {h}')
    ax.set_title(f"u' = u(1 - u/K) with h = {h}:  {len(ts) - 1} steps,  "
                 f"u(1) ≈ {us[-1]:.4f}")
    ax.set_xlabel('t (years)')
    ax.set_ylabel('u (thousands of fish)')
    ax.set_ylim(0.8, 3.1)
    ax.grid(alpha=0.25)
    ax.legend(loc='upper right')
    plt.show()
    plt.close(fig)


interact(
    explore_h,
    h=SelectionSlider(options=[0.5, 0.25, 0.2, 0.125, 0.1, 0.05, 0.04, 0.025, 0.02, 0.01],
                      value=0.25, description='h', continuous_update=False)
);

### What changed, and what stayed the same?

**In your notes,** fill these in from the slider:

At $h=0.5$, $u(1)\approx$ ________. At $h=0.25$, ________. At $h=0.05$, ________. At $h=0.01$,
________.

<details>
<summary>Check yourself after you have written something</summary>

About 1.4844, 1.2679, 1.3069, and 1.3110. With only two steps, the second step starts at
<i>t</i> = 0.5, where the slope is still positive, and walks in a straight line through the whole
winter without ever noticing that the capacity dropped below the population. So it lands high. As <i>h</i> shrinks, the
estimates stop moving much: 1.3069, then 1.3110, crowding toward about 1.31 thousand fish, the same
answer the 3.1 packet reached. The blue path folds onto the dashed curve until you can barely tell
them apart.
</details>

Here is what stayed the same: **the code.** Not one line of `euler` changed while you dragged. One
number did. A hundred steps took the computer less time than it took you to read this sentence.

That matters because this equation has no formula for its solution, so there is nothing to check
the answer against. The only check available is the one you just did: shrink the step and watch
whether the answer stops changing. By hand that check costs an afternoon. In code it costs a drag
of a slider.

## 6. Final challenge: write one yourself

No scaffolding this time beyond the blanks. This is Problem 1 from the 3.1 packet:

$$y'+3y=x^2-3xy+y^2, \qquad y(0)=2.$$

Use Euler's method with $h=0.05$ to estimate $y(0.15)$.

In the cell below, replace every `???` with the right thing, then run it. Three things to think
about before you type:

The equation has to be in the form $y'=\text{something}$ before it can be the slope function.

The variable is called $x$ here instead of $t$. That is fine. The names inside the code are yours to
choose, as long as you use them consistently.

How many steps of $0.05$ does it take to get from $0$ to $0.15$?

If you run the cell before filling in every blank, you will get a `SyntaxError` pointing at a `???`.
That is just Python telling you that you left a blank.

In [ ]:
def f(x, y):
    return ???

x = ???
y = ???
h = ???

for step in range(???):
    y = y + h * f(x, y)
    x = x + h

print('estimate of y(0.15):', round(y, 4))

<details>
<summary>Check yourself after you have run it</summary>

Subtract <code>3y</code> from both sides first, so the slope is
<code>-3*y + x**2 - 3*x*y + y**2</code>. Then <code>x = 0</code>, <code>y = 2</code>,
<code>h = 0.05</code>, and <code>range(3)</code>. The answer is <strong>1.6466</strong>, the same
as the packet.

If you got something else, the usual suspects are a <code>^</code> where a <code>**</code> belongs,
a <code>3xy</code> missing its stars (it should be <code>3*x*y</code>), and the <code>-3*y</code>
that came from moving <code>3y</code> to the other side.
</details>

## Wrap-up

You wrote Euler's method from scratch. Look at how little it took: a function for the slope, three
starting numbers, and a loop with one line of actual mathematics in it. Everything else is
bookkeeping.

That one line, `u = u + h * f(t, u)`, is the update formula

$$u_{n+1}=u_n+h\,f(t_n,u_n)$$

typed into a computer. The spreadsheet column you build for homework \#8 and \#15 is the same thing:
each row of the sheet is one pass through the loop. Python earns its keep when you want a thousand
rows, or when you want to change $h$ by editing one number instead of rebuilding a column.

And the payoff from Section 5 is the one worth remembering. For most equations that show up in the
real world, nobody can write down the solution. A short loop and the habit of shrinking $h$ until the
answer holds still is how those equations actually get answered.

### Exit question

In your notes, copy the loop from Section 3 and circle the one line that **is** Euler's method.

Then, in two or three sentences: what are the other lines in the loop doing, and what would go wrong
if you moved `t = t + h` up to be the **first** line inside the loop?